# Course 12: AI for Accessibility
## Session 6: Continuous signing and sign language translation

**Goal:** train a CTC recognizer on synthetic continuous signing, decode it greedily and with prefix beam search and a gloss language model, and score translations with BLEU and chrF written from scratch.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. By hand first: glosses, CTC and BLEU
* **Glosses.** A gloss is a written label for a sign, usually a word of a spoken language in capitals, such as GO. A signed sentence glossed `TOMORROW I MARKET GO` has how many glosses? A recognizer outputs `TOMORROW I SCHOOL GO NOT`. What is the gloss word error rate (WER)?
* **BLEU** (Papineni et al. 2002) compares a translation with a reference by counting shared word sequences (n-grams). For the reference "আমি কাল বাজারে যাব" ("I will go to the market tomorrow") and the candidate "আমি কাল বাজারে যাব না" ("I will **not** go ..."), what fraction of the candidate's 1-grams, 2-grams, 3-grams and 4-grams appear in the reference?

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["OPENBLAS_NUM_THREADS"] = "1"
import time
import math
import unicodedata
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(1)
t_start = time.time()
np.random.seed(0); torch.manual_seed(0)

def edit_distance(a, b):
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(b) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(b)]

ref, hyp = "TOMORROW I MARKET GO".split(), "TOMORROW I SCHOOL GO NOT".split()
print(f"{len(ref)} glosses; edits {edit_distance(ref, hyp)}; gloss WER {edit_distance(ref, hyp) / len(ref):.2f}")
r_words, c_words = "আমি কাল বাজারে যাব".split(), "আমি কাল বাজারে যাব না".split()
for n in range(1, 5):
    cand = Counter(tuple(c_words[i:i + n]) for i in range(len(c_words) - n + 1))
    refc = Counter(tuple(r_words[i:i + n]) for i in range(len(r_words) - n + 1))
    print(f"{n}-grams: {sum(min(c, refc[g]) for g, c in cand.items())} of {sum(cand.values())} candidate n-grams are in the reference")

### 2. Synthetic continuous signing
In continuous signing there are no pauses between signs: the hands move straight from the end of one sign to the start of the next, and each sign is shaped by its neighbours. This is **co-articulation**. We reuse the 20 synthetic signs and the 16 synthetic signers of session 5 and give the signs made-up English glosses (they are not BdSL signs). Sentences follow a tiny made-up grammar: an optional time sign, a subject, an object, a verb, and an optional NOT, for example `TOMORROW I MARKET GO`. Between two signs the hand moves in 2–4 frames (at 15 frames per second); each sentence starts and ends at a resting position. Signers 0–10 are for training, signer 11 for validation (to tune the decoder), and signers 12–15, never seen, for testing.

In [ ]:
MCP = {"index": (0.30, 1.00), "middle": (0.08, 1.05), "ring": (-0.13, 1.00), "pinky": (-0.32, 0.90)}
LEN = {"index": (0.45, 0.27, 0.20), "middle": (0.50, 0.30, 0.22), "ring": (0.47, 0.28, 0.20), "pinky": (0.37, 0.22, 0.18)}
SPREAD = {"index": 1.0, "middle": 0.3, "ring": -0.4, "pinky": -1.0}
THUMB_CMC, THUMB_LEN, PALM = np.array([0.30, 0.25, 0.0]), (0.45, 0.33, 0.27), np.array([0.0, 0.0, -1.0])

def hand(curl, spread, thumb, size=1.0):
    # session 4's kinematic right hand: 21 landmarks from joint angles
    pts = [np.zeros(3)]
    opp, flex = thumb
    d0 = np.cos(opp) * np.array([0.75, 0.66, 0.0]) + np.sin(opp) * np.array([-0.55, 0.35, -0.76]); d0 /= np.linalg.norm(d0)
    bend = -np.cross(d0, [0, 0, 1.0]); bend /= np.linalg.norm(bend)
    p, th = THUMB_CMC.copy(), 0.0; pts.append(p.copy())
    for L, ang in zip(THUMB_LEN, (0.0, 0.6 * flex, 0.9 * flex)):
        th += ang; p = p + L * (np.cos(th) * d0 + np.sin(th) * bend); pts.append(p.copy())
    for k, name in enumerate(["index", "middle", "ring", "pinky"]):
        a = spread * SPREAD[name] * 0.22
        d = np.array([np.sin(a), np.cos(a), 0.0]); p, th = np.array([*MCP[name], 0.0]), 0.0; pts.append(p.copy())
        for L, ang in zip(LEN[name], (1.45 * curl[k], 1.75 * curl[k], 1.15 * curl[k])):
            th += ang; p = p + L * (np.cos(th) * d + np.sin(th) * PALM); pts.append(p.copy())
    return np.array(pts) * size

def rot(yaw, pitch, roll):
    cy, sy, cp, sp, cr, sr = np.cos(yaw), np.sin(yaw), np.cos(pitch), np.sin(pitch), np.cos(roll), np.sin(roll)
    return (np.array([[cr, -sr, 0], [sr, cr, 0], [0, 0, 1]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
            @ np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]]))

SHAPES = {"B": ([0, 0, 0, 0], 0.0, (1.2, 0.6)), "5": ([0, 0, 0, 0], 1.0, (0.0, 0.1)), "A": ([1, 1, 1, 1], 0.0, (0.15, 0.2)),
          "S": ([1, 1, 1, 1], 0.0, (1.0, 0.9)), "1": ([0, 1, 1, 1], 0.0, (1.0, 0.9)), "V": ([0, 0, 1, 1], 1.0, (1.0, 0.9)),
          "U": ([0, 0, 1, 1], 0.0, (1.0, 0.9)), "Y": ([1, 1, 1, 0], 0.6, (0.0, 0.1)), "L": ([0, 1, 1, 1], 0.0, (0.0, 0.1)),
          "C": ([0.45] * 4, 0.1, (0.7, 0.3))}
LOC = {"forehead": (0.0, 1.1, 0.2), "chin": (0.0, 0.75, 0.25), "chest": (0.0, 0.0, 0.3), "side": (0.6, 0.2, 0.3), "front": (0.2, 0.35, 0.6)}
ORIENT = {"in": (0.0, 0.0, 0.0), "out": (np.pi, 0.0, 0.0), "down": (0.0, -1.2, 0.0), "side": (1.3, 0.0, 0.0)}

def movement(kind, s):
    z = np.zeros_like(s)
    return np.stack({"up": (z, 0.3 * s - 0.15, z), "down": (z, 0.15 - 0.3 * s, z), "sideways": (0.35 * s - 0.17, z, z),
                     "circle": (0.15 * np.cos(2 * np.pi * s), 0.15 * np.sin(2 * np.pi * s), z), "forward": (z, z, 0.3 * s),
                     "tap": (z, z, 0.1 * np.abs(np.sin(4 * np.pi * s))), "shake": (0.08 * np.sin(6 * np.pi * s), z, z),
                     "hold": (z, z, z)}[kind], axis=1)

# (start handshape, end handshape or None, location, movement, palm orientation)
SIGNS = [("B", None, "chin", "forward", "in"), ("B", None, "chest", "forward", "in"), ("B", None, "chin", "down", "in"),
         ("5", None, "chest", "shake", "out"), ("5", None, "front", "circle", "down"), ("A", None, "chest", "circle", "in"),
         ("S", None, "chest", "circle", "in"), ("1", None, "forehead", "sideways", "in"), ("1", None, "front", "up", "out"),
         ("V", None, "front", "down", "out"), ("U", None, "front", "down", "out"), ("Y", None, "side", "shake", "out"),
         ("L", None, "forehead", "forward", "out"), ("C", None, "side", "hold", "side"), ("C", None, "front", "sideways", "side"),
         ("5", "S", "front", "hold", "out"), ("S", "5", "chest", "forward", "out"), ("1", None, "chin", "tap", "in"),
         ("B", None, "side", "tap", "down"), ("Y", None, "front", "circle", "down")]
NAMES = [f"S{k + 1:02d}" for k in range(len(SIGNS))]
rng = np.random.default_rng(0)

def make_signer():
    return dict(speed=rng.uniform(0.7, 1.4), amp=rng.uniform(0.7, 1.3), loc=rng.normal(0, 0.06, 3), size=rng.uniform(0.85, 1.15),
                curl=rng.normal(0, 0.1, 4), spread=rng.normal(0, 0.2), thumb=rng.normal(0, 0.15, 2), lead=rng.integers(2, 9),
                sign_loc=rng.normal(0, 0.08, (20, 3)), sign_amp=rng.uniform(0.75, 1.25, 20), sign_rot=rng.normal(0, 0.25, (20, 3)))

GLOSSES = ["MARKET", "HOME", "EAT", "WATER", "NOW", "I", "YOU", "TOMORROW", "FRIEND", "GO", "SCHOOL", "DRINK", "READ",
           "MOTHER", "TODAY", "FISH", "TEA", "BOOK", "RICE", "NOT"]                 # gloss of synthetic sign S01, S02, ...
gid = {w: i for i, w in enumerate(GLOSSES)}
TIME, SUBJECT = ["TODAY", "TOMORROW", "NOW"], ["I", "YOU", "FRIEND", "MOTHER"]
VERB_OBJECTS = {"GO": ["MARKET", "SCHOOL", "HOME"], "EAT": ["RICE", "FISH"], "DRINK": ["WATER", "TEA"], "READ": ["BOOK"]}

def make_sentence(r):
    s = [r.choice(TIME)] if r.random() < 0.5 else []
    verb = r.choice(list(VERB_OBJECTS))
    s += [r.choice(SUBJECT), r.choice(VERB_OBJECTS[verb]), verb]
    return s + (["NOT"] if r.random() < 0.2 else [])

def sign_frames(sg, k):
    # one sign at 15 frames per second, without the rest pauses of an isolated sign
    hs0, hs1, loc, mv, ori = SIGNS[k]
    n = int(round(20 / (sg["speed"] * rng.uniform(0.85, 1.15))))
    s = np.linspace(0, 1, n)
    path = movement(mv, s) * sg["amp"] * sg["sign_amp"][k] * rng.uniform(0.9, 1.1)
    base = np.array(LOC[loc]) + sg["loc"] + sg["sign_loc"][k]
    R = rot(*ORIENT[ori]) @ rot(*sg["sign_rot"][k]) @ rot(*rng.normal(0, 0.1, 3))
    styled = lambda c, sp, th: hand([a + o for a, o in zip(c, sg["curl"])], sp + sg["spread"],
                                    tuple(a + o for a, o in zip(th, sg["thumb"])), sg["size"])
    P0 = styled(*SHAPES[hs0]); P1 = styled(*SHAPES[hs1]) if hs1 else P0
    w = np.clip((s - 0.3) / 0.4, 0, 1)[:, None, None] if hs1 else np.zeros((n, 1, 1))
    return base + path[:, None, :] + 0.25 * ((1 - w) * P0 + w * P1) @ R.T

def sign_sentence(sg, glosses, trans=(2, 5)):
    rest = np.array([0.1, -0.6, 0.3]) + 0.25 * hand(*SHAPES["B"], sg["size"]) @ rot(0, -1.2, 0).T
    parts, spans, prev, t = [np.repeat(rest[None], rng.integers(2, 6), 0)], [], rest[None], 0
    t = len(parts[0])
    for w in glosses:
        X = sign_frames(sg, gid[w])
        a = np.linspace(0, 1, rng.integers(*trans) + 2)[1:-1][:, None, None]      # transition: co-articulation
        parts += [(1 - a) * prev[-1] + a * X[0], X]
        t += len(a); spans.append((t, t + len(X))); t += len(X); prev = X
    a = np.linspace(0, 1, 5)[1:-1][:, None, None]
    parts += [(1 - a) * prev[-1] + a * rest, np.repeat(rest[None], rng.integers(2, 6), 0)]
    X = np.concatenate(parts) + rng.normal(0, 0.01, (sum(len(p) for p in parts), 21, 3))
    for i in np.where(rng.random(len(X)) < 0.05)[0]:
        if i > 0:
            X[i] = X[i - 1]
    return X.astype(np.float32), spans

signers = [make_signer() for _ in range(16)]
sent_rng = np.random.default_rng(1)
data, labels, spans, g = [], [], [], []
for si, sg in enumerate(signers):
    for _ in range(50):
        s = make_sentence(sent_rng)
        X, sp = sign_sentence(sg, s)
        data.append(X); labels.append(s); spans.append(sp); g.append(si)
g = np.array(g); lens = np.array([len(d) for d in data])
train = [i for i in range(len(data)) if g[i] <= 10]; val = [i for i in range(len(data)) if g[i] == 11]
test = [i for i in range(len(data)) if g[i] >= 12]
print(f"{len(data)} sentences, {lens.min()} to {lens.max()} frames (mean {lens.mean():.0f}); "
      f"{np.mean([len(l) for l in labels]):.2f} glosses per sentence; train/val/test {len(train)}/{len(val)}/{len(test)}")
print("example:", " ".join(labels[0]), "| frames of each sign:", spans[0])

### 3. A CTC recognizer over glosses
The same recipe as in session 2, with glosses in place of digits: a convolution over time with stride 2, a bidirectional GRU, and CTC over 21 symbols (the blank and 20 glosses). CTC needs no sign boundaries, only the gloss sequence of each sentence.

In [ ]:
flat = np.concatenate([d.reshape(len(d), -1) for d in data]); MU, SD = flat.mean(0), flat.std(0) + 1e-6

def batch(idx):
    xs = [(data[i].reshape(len(data[i]), -1) - MU) / SD for i in idx]
    L = [len(x) for x in xs]; T = max(L)
    X = np.array([np.concatenate([x, np.repeat(x[-1:], T - len(x), 0)]) for x in xs])
    targets = torch.tensor([gid[w] + 1 for i in idx for w in labels[i]])
    return torch.tensor(X, dtype=torch.float32), torch.tensor(L), targets, torch.tensor([len(labels[i]) for i in idx])

class GlossRecognizer(nn.Module):
    def __init__(self, h=64, n=21):
        super().__init__()
        self.conv = nn.Conv1d(63, h, 5, stride=2, padding=2)
        self.gru = nn.GRU(h, h, batch_first=True, bidirectional=True); self.out = nn.Linear(2 * h, n)

    def forward(self, x, L):
        z = torch.relu(self.conv(x.transpose(1, 2))).transpose(1, 2); L2 = (L - 1) // 2 + 1
        z, _ = self.gru(nn.utils.rnn.pack_padded_sequence(z, L2, batch_first=True, enforce_sorted=False))
        z, _ = nn.utils.rnn.pad_packed_sequence(z, batch_first=True)
        return self.out(z).log_softmax(-1), L2

def log_probs(model, idx):
    model.eval()
    with torch.no_grad():
        x, L, _, _ = batch(idx); lp, L2 = model(x, L)
    model.train()
    return [lp[k, :L2[k]].numpy() for k in range(len(idx))]

def greedy(lp):
    best = lp.argmax(-1)
    return [GLOSSES[k - 1] for i, k in enumerate(best) if k != 0 and (i == 0 or k != best[i - 1])]

def gloss_wer(refs, hyps):
    return sum(edit_distance(r, h) for r, h in zip(refs, hyps)) / sum(len(r) for r in refs)

torch.manual_seed(0)
model = GlossRecognizer(); opt = torch.optim.Adam(model.parameters(), lr=3e-3); ctc = nn.CTCLoss(zero_infinity=True)
order_rng = np.random.default_rng(3)
refs_test = [labels[i] for i in test]
for epoch in range(20):
    order = order_rng.permutation(train)
    for b in range(0, len(order), 16):
        x, L, tg, tl = batch(order[b:b + 16]); lp, L2 = model(x, L)
        loss = ctc(lp.transpose(0, 1), tg, L2, tl)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    if epoch % 5 == 4:
        print(f"epoch {epoch + 1}: last batch CTC loss {loss.item():.3f}; test gloss WER (greedy) "
              f"{gloss_wer(refs_test, [greedy(l) for l in log_probs(model, test)]):.3f} ({time.time() - t_start:.0f} s)")

### 4. Greedy decoding against prefix beam search, with and without a language model
Greedy decoding keeps one path. **Prefix beam search** (explained in Hannun 2017) keeps the best `beam` output prefixes at each frame and adds up the probabilities of all frame paths that give the same prefix, ending in a blank or not. It can also add a **language model** (LM) score: here a gloss bigram model counted from the training sentences, with add-one smoothing, weighted by $\alpha$, plus a bonus $\beta$ per gloss so that the LM does not simply favour short outputs. We choose $\alpha$ and $\beta$ on the validation signer, then test once.

In [ ]:
bigram = defaultdict(Counter)
for i in train:
    seq = ["<s>"] + labels[i] + ["</s>"]
    for a, b in zip(seq[:-1], seq[1:]):
        bigram[a][b] += 1

def lm_logp(prev, nxt):
    return math.log((bigram[prev][nxt] + 1) / (sum(bigram[prev].values()) + len(GLOSSES) + 1))

def prefix_beam_search(lp, beam=8, alpha=0.0, beta=0.0):
    beams = {(): (0.0, -np.inf)}                                                 # prefix -> (log P ending in blank, log P ending in a gloss)
    for t in range(len(lp)):
        nxt = defaultdict(lambda: [-np.inf, -np.inf])
        cands = [c for c in range(1, lp.shape[1]) if lp[t, c] > -9]             # skip glosses with probability below about 1e-4
        for prefix, (pb, pnb) in beams.items():
            total = np.logaddexp(pb, pnb)
            e = nxt[prefix]; e[0] = np.logaddexp(e[0], total + lp[t, 0])          # a blank keeps the prefix
            last = prefix[-1] if prefix else "<s>"
            for c in cands:
                w = GLOSSES[c - 1]
                bonus = alpha * lm_logp(last, w) + beta
                if w == last:
                    e = nxt[prefix]; e[1] = np.logaddexp(e[1], pnb + lp[t, c])    # repeated gloss without a blank: same prefix
                    e2 = nxt[prefix + (w,)]; e2[1] = np.logaddexp(e2[1], pb + lp[t, c] + bonus)   # after a blank: a new gloss
                else:
                    e2 = nxt[prefix + (w,)]; e2[1] = np.logaddexp(e2[1], total + lp[t, c] + bonus)
        beams = dict(sorted(nxt.items(), key=lambda kv: -np.logaddexp(*kv[1]))[:beam])
    end = lambda kv: np.logaddexp(*kv[1]) + alpha * lm_logp(kv[0][-1] if kv[0] else "<s>", "</s>")
    return list(max(beams.items(), key=end)[0])

LP_val, LP_test = log_probs(model, val), log_probs(model, test)
refs_val = [labels[i] for i in val]
grid = {(a, b): gloss_wer(refs_val, [prefix_beam_search(l, alpha=a, beta=b) for l in LP_val]) for a in (0.3, 0.6, 1.0) for b in (0.0, 1.0, 2.0)}
alpha, beta = min(grid, key=grid.get)
print("validation gloss WER for (alpha, beta):", {k: round(v, 3) for k, v in grid.items()})
results = {"greedy": [greedy(l) for l in LP_test],
           "prefix beam search (beam 8)": [prefix_beam_search(l) for l in LP_test],
           f"beam + bigram LM (alpha {alpha}, beta {beta})": [prefix_beam_search(l, alpha=alpha, beta=beta) for l in LP_test]}
for name, hyps in results.items():
    errs = [edit_distance(r, h) for r, h in zip(refs_test, hyps)]
    print(f"{name:40s}: test gloss WER {gloss_wer(refs_test, hyps):.3f}; sentences fully correct {np.mean([e == 0 for e in errs]):.3f}")
for name, hyps in results.items():
    print(f"  per test signer, {name.split(' (')[0]:28s}: " +
          ", ".join(f"{s}: {gloss_wer([labels[i] for i in test if g[i] == s], [h for i, h in zip(test, hyps) if g[i] == s]):.3f}" for s in range(12, 16)))

k = next(j for j, (r, h) in enumerate(zip(refs_test, results["greedy"])) if r != h)
print("a sentence greedy got wrong:  reference:", " ".join(refs_test[k]), "| greedy:", " ".join(results["greedy"][k]),
      "| beam + LM:", " ".join(list(results.values())[2][k]))

i0 = test[k]; X = data[i0]; lp = LP_test[k]
fig, ax = plt.subplots(2, 1, figsize=(9, 3.5), sharex=True)
speed = np.r_[0, np.linalg.norm(np.diff(X[:, 0], axis=0), axis=1)] * 15
ax[0].plot(np.arange(len(X)) / 15, X[:, 0, 1], label="wrist height"); ax[0].plot(np.arange(len(X)) / 15, speed / 4, label="wrist speed / 4")
for (a_, b_), w in zip(spans[i0], labels[i0]):
    ax[0].axvspan(a_ / 15, b_ / 15, color="0.85"); ax[0].text((a_ + b_) / 30, ax[0].get_ylim()[1] * 0.9, w, ha="center", fontsize=7)
ax[0].set_ylabel("shoulder widths"); ax[0].legend(fontsize=7, loc="lower right"); ax[0].set_title("one test sentence: grey = true sign spans", fontsize=9)
tt = (np.arange(len(lp)) * 2 + 0.5) / 15
ax[1].plot(tt, np.exp(lp[:, 0]), color="0.6", lw=1, label="blank")
for c in np.unique(lp.argmax(1)):
    if c:
        ax[1].plot(tt, np.exp(lp[:, c]), lw=1.5, label=GLOSSES[c - 1])
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("probability"); ax[1].legend(fontsize=6, ncol=4, loc="center right")
plt.tight_layout(); plt.show()

### 5. Translation is not recognition: BLEU and chrF
A translation system must produce a sentence in a spoken language, with its own word order, inflections and idioms. Glosses `TOMORROW I MARKET GO` become "আমি কাল বাজারে যাব": the subject moves first, বাজার takes the locative ending -এ, and the verb takes the future first-person form. In everyday Bangla you even "eat" water and tea (পানি খাওয়া, চা খাওয়া), so DRINK is not always one fixed word.

We write two common metrics from scratch:
* **BLEU** (Papineni et al. 2002): the geometric mean of the clipped n-gram precisions for n = 1 to 4, times a brevity penalty $\min(1, e^{1 - r/c})$ for a candidate of length c shorter than the reference length r. For single sentences we smooth zero counts as sacreBLEU's "exp" method does: the k-th order with no matches gets precision $1/(2^k \cdot \text{total})$.
* **chrF** (Popović 2015): an F-score over character n-grams (n = 1 to 6, spaces removed), with recall weighted more than precision ($\beta = 2$).

Then we score four candidate translations of one reference.

In [ ]:
def ngrams(seq, n):
    return Counter(tuple(seq[i:i + n]) for i in range(len(seq) - n + 1))

def bleu(hyps, refs, smooth=False):
    # corpus BLEU on whitespace tokens, 0-100; with smooth=True, the 'exp' smoothing used for single sentences
    correct, total, c_len, r_len = [0] * 4, [0] * 4, 0, 0
    for h, r in zip(hyps, refs):
        h, r = h.split(), r.split(); c_len += len(h); r_len += len(r)
        for n in range(1, 5):
            hc, rc = ngrams(h, n), ngrams(r, n)
            correct[n - 1] += sum(min(c, rc[gram]) for gram, c in hc.items()); total[n - 1] += sum(hc.values())
    if not any(correct):
        return 0.0
    logs, k = [], 1
    for n in range(4):
        if correct[n] > 0:
            logs.append(math.log(correct[n] / total[n]))
        elif smooth and total[n] > 0:
            k *= 2; logs.append(math.log(1 / (k * total[n])))
        else:
            return 0.0
    bp = 1.0 if c_len >= r_len else math.exp(1 - r_len / c_len)
    return 100 * bp * math.exp(sum(logs) / 4)

def chrf(hyp, ref, max_n=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    precs, recs = [], []
    for n in range(1, max_n + 1):
        hc, rc = ngrams(h, n), ngrams(r, n)
        if hc and rc:
            m = sum(min(c, rc[gram]) for gram, c in hc.items())
            precs.append(m / sum(hc.values())); recs.append(m / sum(rc.values()))
    P, R = np.mean(precs), np.mean(recs)
    return 0.0 if P + R == 0 else 100 * (1 + beta ** 2) * P * R / (beta ** 2 * P + R)

nfc = lambda s: unicodedata.normalize("NFC", s)
REF = nfc("আমি কাল বাজারে যাব")                                                # "I will go to the market tomorrow"
CANDIDATES = {"word-for-word from the glosses": "কাল আমি বাজার যাওয়া",            # ungrammatical: dictionary forms
              "correct paraphrase": "আগামীকাল আমি বাজারে যাব",                     # "Tomorrow I will go to the market"
              "wrong: negated": "আমি কাল বাজারে যাব না",                             # "I will NOT go to the market tomorrow"
              "wrong: other place": "আমি কাল স্কুলে যাব"}                           # "I will go to school tomorrow"
scores = {}
for name, c in CANDIDATES.items():
    scores[name] = (bleu([nfc(c)], [REF], smooth=True), chrf(nfc(c), REF))
    print(f"{name:32s}: sentence BLEU {scores[name][0]:5.1f}   chrF {scores[name][1]:5.1f}")

fig, ax = plt.subplots(figsize=(8, 2.6))
names = list(scores); yy = np.arange(len(names))
ax.barh(yy - 0.2, [scores[n][0] for n in names], 0.4, label="BLEU"); ax.barh(yy + 0.2, [scores[n][1] for n in names], 0.4, label="chrF")
ax.set_yticks(yy); ax.set_yticklabels(names, fontsize=8); ax.set_xlabel("score (0-100)"); ax.set_xlim(0, 100)
ax.set_title("four candidate translations of one reference sentence", fontsize=9); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"runtime so far {time.time() - t_start:.0f} s")

### 6. Checking our metrics against sacreBLEU (NOT RUN here)
sacreBLEU (Post 2018) is the standard implementation of BLEU and chrF; report its signature string with any score so that others can reproduce it. Our sentence BLEU uses the same "exp" smoothing and our chrF the same averaging, so the numbers should agree for these unpunctuated sentences.

In [ ]:
# NOT RUN: needs the sacrebleu package (pip install sacrebleu), which is not installed here
import sacrebleu
for name, c in CANDIDATES.items():
    b = sacrebleu.sentence_bleu(nfc(c), [REF]).score
    ch = sacrebleu.sentence_chrf(nfc(c), [REF]).score
    print(f"{name:32s}: sacreBLEU BLEU {b:5.1f} (ours {scores[name][0]:5.1f}); chrF {ch:5.1f} (ours {scores[name][1]:5.1f})")

### 7. Exercises

In [ ]:
# TODO: gloss-to-text by rules (a tiny Sign2Gloss2Text). Write gloss_to_bangla(glosses) for the grammar of section 2:
# subject first, then time, object (with -এ after MARKET and SCHOOL), verb; future tense after TODAY or TOMORROW,
# present otherwise; add না after the verb for NOT. Ask a Bangla speaker to check ten outputs. Then translate the
# greedy and the beam + LM test outputs and report corpus BLEU against the translations of the true glosses.
def gloss_to_bangla(glosses):
    return None          # replace with your code: return a Bangla sentence

r = gloss_to_bangla(["TOMORROW", "I", "MARKET", "GO"])
print("not done yet" if r is None else r)

In [ ]:
# TODO: faster signing. Generate 50 new sentences for each test signer with transitions of only 1 frame
# (sign_sentence(sg, glosses, trans=(1, 2))), score the trained model with greedy decoding and with beam + LM, and
# compare with the results above. Which errors grow: substitutions, deletions or insertions?
def fast_signing_wer():
    return None          # replace with your code: return (greedy WER, beam + LM WER)

r = fast_signing_wer()
print("not done yet" if r is None else r)

### What to notice
(to be written from the outputs)